# Comparação de Fronteiras de Pareto — VRF-NBI, CNBI, NSGA-III, MOEA/D vs. $P^*$

Este notebook compara, sob a mesma referência de qualidade ($P^*$, a fronteira
empírica de 8 objetivos gerada em `notebook_referencia_pareto_8D.ipynb`), as
fronteiras obtidas por quatro estratégias distintas:

- **VRF-NBI** — 66 pontos, Simplex-Lattice {3,10} sobre 3 fatores rotacionados (`VRF_Pareto.xlsx`)
- **CNBI** — fronteira combinatória completa (`RSM-CNBI.ipynb`)
- **NSGA-III** — mesmo orçamento de avaliações do RSM que o CNBI (múltiplas seeds)
- **MOEA/D** — idem

As métricas principais são GD, IGD, hipervolume, **Spacing** e **Sparsity**,
sempre no espaço dos objetivos orientado para minimização e normalizado pela
referência $P^*$.

Na segunda parte, o CNBI é reduzido à mesma cardinalidade do VRF-NBI (66
pontos) usando seis métodos de subamostragem: Farthest-Point Sampling,
K-Medoids, MiniBatchKMeans, GMM, Direções de Referência e
**Clusterização Hierárquica**. Isso permite uma comparação de cardinalidade
equalizada contra o VRF-NBI.

**Arquivos necessários** (coloque na mesma pasta deste notebook, ou ajuste os
caminhos no preset abaixo):
- `referencia_pareto_8D.csv` ($P^*$)
- `VRF_Pareto.xlsx`
- `fronteira_ND_cnbi_..._piso....csv` (saída do RSM-CNBI)
- `nsga3_seed*.csv` e `moead_seed*.csv` (saída do RSM-CNBI_orcamento, seção de NSGA-III/MOEA-D) — **opcionais**


In [ ]:
# ============================================================
# 0) PRESETS — ajuste os caminhos conforme sua pasta de trabalho
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import cdist
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

CAMINHO_PSTAR = Path("referencia_pareto_8D.csv")
CAMINHO_VRF = Path("VRF_Pareto.xlsx")
CAMINHO_CNBI = Path("fronteira_ND_cnbi.csv")
DIR_EAS = Path(".")  # pasta onde procurar nsga3_seed*.csv / moead_seed*.csv

RESP = ['T', 'MTTF', 'WR', 'Ra', 'Rt', 'Kp', 'ROI', 'OEE']
DIRECAO = {'T': 'max', 'MTTF': 'max', 'WR': 'min', 'Ra': 'min', 'Rt': 'min',
           'Kp': 'min', 'ROI': 'max', 'OEE': 'max'}
SIGNS = np.array([-1 if DIRECAO[c] == 'max' else 1 for c in RESP])

OUT_DIR = Path("saida_comparacao")
OUT_DIR.mkdir(exist_ok=True, parents=True)

print("Presets carregados.")


## 1) Carregamento das fronteiras

In [ ]:
# ============================================================
# 1) CARREGAMENTO DAS FRONTEIRAS
# ============================================================
Pstar = pd.read_csv(CAMINHO_PSTAR)[RESP].to_numpy(dtype=float)

vrf_df = pd.read_excel(CAMINHO_VRF, header=0, usecols='A:R')
Fvrf = vrf_df[RESP].to_numpy(dtype=float)

cnbi_df = pd.read_csv(CAMINHO_CNBI)
Fcnbi = cnbi_df[[f'{c}_fisico' for c in RESP]].to_numpy(dtype=float)

print(f"P*   : {Pstar.shape[0]} pontos")
print(f"VRF  : {Fvrf.shape[0]} pontos")
print(f"CNBI : {Fcnbi.shape[0]} pontos")


def carregar_ea_seeds(prefixo):
    """Carrega todas as fronteiras nsga3_seed*.csv / moead_seed*.csv, se existirem.
    Retorna (lista_de_arrays, lista_de_arquivos); (None, []) se nada for encontrado.
    """
    arquivos = sorted(DIR_EAS.glob(f"{prefixo}_seed*.csv"))
    if not arquivos:
        return None, []
    frentes = []
    for arq in arquivos:
        d = pd.read_csv(arq)
        if all(c in d.columns for c in RESP):
            frentes.append(d[RESP].to_numpy(dtype=float))
    return frentes, arquivos


frentes_nsga3, arquivos_nsga3 = carregar_ea_seeds("nsga3")
frentes_moead, arquivos_moead = carregar_ea_seeds("moead")

if frentes_nsga3:
    print(f"NSGA-III: {len(frentes_nsga3)} seeds encontradas "
          f"({[len(f) for f in frentes_nsga3]} pontos cada)")
else:
    print("NSGA-III: nenhum arquivo encontrado — rode a seção de NSGA-III/MOEA-D "
          "do notebook RSM-CNBI_orcamento.ipynb primeiro (requer pymoo). "
          "As seções que dependem disso serão puladas automaticamente.")

if frentes_moead:
    print(f"MOEA/D: {len(frentes_moead)} seeds encontradas "
          f"({[len(f) for f in frentes_moead]} pontos cada)")
else:
    print("MOEA/D: nenhum arquivo encontrado — mesma observação acima.")


## 2) Normalização e métricas GD / IGD / HV / Spacing / Sparsity

Ideal e nadir são calculados a partir de $P^*$, garantindo que todas as
fronteiras sejam comparadas na mesma régua normalizada.

- **Spacing**: desvio-padrão das distâncias ao vizinho mais próximo. Quanto
  menor, mais uniforme é o espaçamento local.
- **Sparsity**: distância média ao vizinho mais próximo. Quanto menor, mais
  densa é a frente.

A Sparsity é sensível à cardinalidade. Por isso, sua interpretação mais justa
ocorre na seção de redução, onde todas as fronteiras têm o mesmo número de
pontos.


In [ ]:
# ============================================================
# 2) NORMALIZAÇÃO E MÉTRICAS GD / IGD / HV / SPACING / SPARSITY
# ============================================================
Pstar_min = Pstar * SIGNS
ideal = Pstar_min.min(axis=0)
nadir = Pstar_min.max(axis=0)
amplitude = np.maximum(nadir - ideal, 1e-12)


def normalizar(F):
    return (F * SIGNS - ideal) / amplitude


def GD(A, ref):
    """Generational Distance (p=1): média das distâncias de A à referência.

    A média simples evita o viés de cardinalidade da formulação
    sqrt(sum(d²))/|A|.
    """
    d = cdist(A, ref).min(axis=1)
    return float(d.mean())


def IGD(A, ref):
    """Inverted Generational Distance: convergência + cobertura de A."""
    d = cdist(ref, A).min(axis=1)
    return float(d.mean())


def distancias_vizinho_mais_proximo(F):
    """Distância euclidiana ao vizinho mais próximo sem formar matriz n×n.

    O cKDTree evita o alto consumo de memória que cdist(F, F) teria para P*.
    """
    F = np.asarray(F, dtype=float)
    F = F[np.all(np.isfinite(F), axis=1)]
    if len(F) < 2:
        return np.array([], dtype=float)
    arvore = cKDTree(F)
    distancias, _ = arvore.query(F, k=2)
    return np.asarray(distancias[:, 1], dtype=float)


def Spacing(F):
    """Desvio-padrão das distâncias ao vizinho mais próximo.

    Menor valor indica distribuição local mais uniforme.
    """
    d = distancias_vizinho_mais_proximo(F)
    if len(d) <= 2:
        return np.nan
    return float(np.sqrt(np.mean((d - d.mean()) ** 2)))


def Sparsity(F):
    """Distância média ao vizinho mais próximo.

    Menor valor indica frente mais densa; a métrica é sensível ao número de
    pontos e deve ser priorizada em comparações de cardinalidade equalizada.
    """
    d = distancias_vizinho_mais_proximo(F)
    if len(d) == 0:
        return np.nan
    return float(d.mean())


def mascara_nao_dominados(F):
    """Máscara booleana True para pontos não-dominados (minimização)."""
    F = np.asarray(F, dtype=float)
    n = F.shape[0]
    is_nd = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_nd[i]:
            continue
        dom = np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)
        dom[i] = False
        if dom.any():
            is_nd[i] = False
    return is_nd


# ------------------------------------------------------------------
# Hipervolume por Monte Carlo (espaço normalizado, minimização)
# ------------------------------------------------------------------
def gerar_amostra_hv(m, z_ref=1.1, n=50_000, seed=2026):
    """Amostra uniforme COMUM em [0, z_ref]^m.

    A mesma amostra é usada para todos os métodos (Monte Carlo pareado).
    """
    rng = np.random.default_rng(seed)
    return rng.uniform(0.0, z_ref, size=(n, m))


def HV_mc(A, Z, z_ref=1.1, chunk=2_000):
    """Hipervolume estimado: fração de Z dominada por A × volume da caixa."""
    A = np.asarray(A, dtype=float)
    A = A[mascara_nao_dominados(A)]
    A = A[np.all(A <= z_ref, axis=1)]
    if len(A) == 0:
        return 0.0
    dominado = np.zeros(len(Z), dtype=bool)
    for ini in range(0, len(Z), chunk):
        Zc = Z[ini:ini + chunk]
        dom_c = (Zc[:, None, :] >= A[None, :, :]).all(axis=2).any(axis=1)
        dominado[ini:ini + chunk] = dom_c
    vol_caixa = float(z_ref) ** Z.shape[1]
    return float(dominado.mean() * vol_caixa)


Pstar_n = normalizar(Pstar)
Fvrf_n = normalizar(Fvrf)
Fcnbi_n = normalizar(Fcnbi)

print("Normalização concluída (ideal/nadir extraídos de P*).")


## 3) Comparação principal: VRF-NBI vs. CNBI vs. NSGA-III vs. MOEA/D vs. $P^*$

Para NSGA-III e MOEA/D (estocásticos, múltiplas seeds), reporta-se tanto a
**mediana das seeds** quanto o **pool de todas as seeds filtrado para
não-dominados**.

GD, IGD e HV avaliam convergência/cobertura. Spacing avalia uniformidade local,
enquanto Sparsity quantifica a densidade local e deve ser lida com cautela
quando as cardinalidades são diferentes.


In [ ]:
# ============================================================
# 3) COMPARAÇÃO PRINCIPAL — GD, IGD, HV, SPACING E SPARSITY
# ============================================================
Z_HV = gerar_amostra_hv(len(RESP))


def metricas_frente(Fn):
    """Calcula todas as métricas no espaço normalizado."""
    return {
        "GD": GD(Fn, Pstar_n),
        "IGD": IGD(Fn, Pstar_n),
        "HV": HV_mc(Fn, Z_HV),
        "Spacing": Spacing(Fn),
        "Sparsity": Sparsity(Fn),
    }


linhas_principal = [
    {"metodo": "VRF-NBI", "n": len(Fvrf_n), **metricas_frente(Fvrf_n)},
    {"metodo": "CNBI (completo)", "n": len(Fcnbi_n), **metricas_frente(Fcnbi_n)},
    {
        "metodo": "P* (teto do HV)",
        "n": len(Pstar_n),
        "GD": 0.0,
        "IGD": 0.0,
        "HV": HV_mc(Pstar_n, Z_HV),
        "Spacing": Spacing(Pstar_n),
        "Sparsity": Sparsity(Pstar_n),
    },
]

df_ea_seeds_long = []

for nome, frentes, arquivos in [
    ("NSGA-III", frentes_nsga3, arquivos_nsga3),
    ("MOEA/D", frentes_moead, arquivos_moead),
]:
    if not frentes:
        continue

    metricas_seeds = []
    for i, F in enumerate(frentes):
        Fn = normalizar(F)
        met = metricas_frente(Fn)
        metricas_seeds.append(met)
        df_ea_seeds_long.append({
            "metodo": nome,
            "seed": i + 1,
            "n": len(F),
            **met,
        })

    df_met_seeds = pd.DataFrame(metricas_seeds)

    F_pool = np.vstack(frentes)
    Fn_pool = normalizar(F_pool)
    nd_mask = mascara_nao_dominados(Fn_pool)
    Fn_pool_nd = Fn_pool[nd_mask]

    linhas_principal.append({
        "metodo": f"{nome} (mediana das seeds)",
        "n": int(np.median([len(f) for f in frentes])),
        **{
            coluna: float(df_met_seeds[coluna].median())
            for coluna in ["GD", "IGD", "HV", "Spacing", "Sparsity"]
        },
    })
    linhas_principal.append({
        "metodo": f"{nome} (pool de seeds, não-dominado)",
        "n": int(nd_mask.sum()),
        **metricas_frente(Fn_pool_nd),
    })

df_resultado_principal = pd.DataFrame(linhas_principal)
df_ea_seeds_long = pd.DataFrame(df_ea_seeds_long)

display(df_resultado_principal)
df_resultado_principal.to_csv(
    OUT_DIR / "comparacao_principal_metricas.csv", index=False
)


## 4) Figuras — comparação principal

In [ ]:
# ============================================================
# FIGURA 1 — métricas por método
# ============================================================
def cor_metodo(m):
    if "VRF" in m:
        return "#4C72B0"
    if "CNBI" in m:
        return "#DD8452"
    if "NSGA" in m:
        return "#55A868"
    if "MOEA" in m:
        return "#C44E52"
    return "#8172B2"


metricas_fig1 = [
    ("GD", "GD vs P* (menor é melhor)"),
    ("IGD", "IGD vs P* (menor é melhor)"),
    ("Spacing", "Spacing (menor = mais uniforme)"),
    ("Sparsity", "Sparsity (menor = mais densa)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cores = [cor_metodo(m) for m in df_resultado_principal["metodo"]]

for ax, (metrica, titulo) in zip(axes.ravel(), metricas_fig1):
    ax.barh(
        df_resultado_principal["metodo"],
        df_resultado_principal[metrica],
        color=cores,
    )
    ax.set_title(titulo)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_metricas_principais.png", dpi=150)
plt.show()


In [ ]:
# ============================================================
# DIAGNÓSTICO DOS X DAS SEEDS DO MOEA/D
# ATENÇÃO: versões do RSM-CNBI_orcamento anteriores à correção exportavam
# o genótipo X NÃO projetado, enquanto F era avaliado no ponto PROJETADO
# na esfera. Este diagnóstico alerta se detectar X inviável — nesse caso,
# re-exporte as seeds com o notebook corrigido antes de usar os X
# (os valores de F e as métricas GD/IGD/HV não são afetados).
# ============================================================
from pathlib import Path

FACTOR_COLS = ["cs", "f", "md"]
RAIO_DOE = 2 ** 0.75   # raio axial exato do CCD rotacional (k=3) = 1.6817928...
TOL_ESFERA = 1e-6

for seed_file in sorted(Path(".").glob("moead_seed*.csv")):
    d = pd.read_csv(seed_file)
    faltando = [c for c in FACTOR_COLS if c not in d.columns]
    if faltando:
        print(f"{seed_file.name}: faltam colunas {faltando} — pulando")
        continue
    X = d[FACTOR_COLS].to_numpy(dtype=float)
    normas = np.linalg.norm(X, axis=1)
    pct_borda = (np.abs(normas - RAIO_DOE) < 1e-3).mean()
    n_inviaveis = int((normas > RAIO_DOE + TOL_ESFERA).sum())
    msg = (f"{seed_file.name}: dispersão de x = {X.std(axis=0)}, "
           f"% na borda da esfera = {pct_borda:.1%}")
    if n_inviaveis > 0:
        msg += (f"\n    [AVISO: {n_inviaveis} ponto(s) com ||x|| > raio — "
                f"arquivo gerado ANTES da correção da exportação do MOEA/D; "
                f"re-exporte as seeds no RSM-CNBI_orcamento corrigido]")
    print(msg)


In [ ]:
# ============================================================
# FIGURA 2 — variabilidade entre seeds dos EAs
# ============================================================
if not df_ea_seeds_long.empty:
    metricas_boxplot = ["GD", "IGD", "Spacing", "Sparsity"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for ax, metrica in zip(axes.ravel(), metricas_boxplot):
        grupos = df_ea_seeds_long["metodo"].unique()
        dados = [
            df_ea_seeds_long.loc[
                df_ea_seeds_long["metodo"] == m, metrica
            ].values
            for m in grupos
        ]
        ax.boxplot(dados, labels=grupos)
        ax.axhline(
            df_resultado_principal.loc[
                df_resultado_principal["metodo"] == "VRF-NBI", metrica
            ].values[0],
            color="#4C72B0",
            linestyle="--",
            label="VRF-NBI",
        )
        ax.axhline(
            df_resultado_principal.loc[
                df_resultado_principal["metodo"] == "CNBI (completo)", metrica
            ].values[0],
            color="#DD8452",
            linestyle="--",
            label="CNBI",
        )
        ax.set_title(f"Variabilidade entre seeds — {metrica}")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / "fig2_variabilidade_seeds.png", dpi=150)
    plt.show()
else:
    print("Sem arquivos de seeds: boxplots não gerados.")


In [ ]:
# ============================================================
# FIGURA 3 — cobertura em pares de objetivos (unidades físicas)
# ============================================================

pares = [
    ("T", "Kp"),
    ("Ra", "ROI"),
    ("WR", "OEE"),
    ("MTTF", "Rt"),
]

df_pstar_full = pd.DataFrame(Pstar, columns=RESP)
df_vrf_full = pd.DataFrame(Fvrf, columns=RESP)
df_cnbi_full = pd.DataFrame(Fcnbi, columns=RESP)

# Pools dos métodos evolucionários
df_n = (
    pd.DataFrame(np.vstack(frentes_nsga3), columns=RESP)
    if frentes_nsga3
    else None
)

df_m = (
    pd.DataFrame(np.vstack(frentes_moead), columns=RESP)
    if frentes_moead
    else None
)

import math

ncols = 2
nrows = math.ceil(len(pares) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(12, 5 * nrows),
)

axes = np.asarray(axes).ravel()

# Plota cada dupla de objetivos
for ax, (ox, oy) in zip(axes, pares):

    ax.scatter(
        df_pstar_full[ox],
        df_pstar_full[oy],
        s=6,
        color="lightgray",
        alpha=0.5,
        label="P*",
    )

    ax.scatter(
        df_cnbi_full[ox],
        df_cnbi_full[oy],
        s=14,
        color="#DD8452",
        alpha=0.7,
        label="CNBI",
    )

    ax.scatter(
        df_vrf_full[ox],
        df_vrf_full[oy],
        s=22,
        color="#4C72B0",
        alpha=0.9,
        label="VRF-NBI",
    )

    if df_n is not None:
        ax.scatter(
            df_n[ox],
            df_n[oy],
            s=10,
            color="#55A868",
            alpha=0.5,
            label="NSGA-III",
        )

    if df_m is not None:
        ax.scatter(
            df_m[ox],
            df_m[oy],
            s=10,
            color="#C44E52",
            alpha=0.5,
            label="MOEA/D",
        )

    ax.set_xlabel(ox)
    ax.set_ylabel(oy)
    ax.set_title(f"{ox} vs {oy}")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

# Remove eixos vazios quando a quantidade de pares não fecha a grade
for ax in axes[len(pares):]:
    ax.remove()

plt.tight_layout()
plt.savefig(
    OUT_DIR / "fig3_cobertura_objetivos.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 5) Redução do CNBI para a cardinalidade do VRF-NBI (66 pontos)

Reaplica seis métodos de subamostragem para permitir uma comparação de
cardinalidade equalizada contra o VRF-NBI. A antiga seleção por Crowding
Distance foi substituída por **clusterização hierárquica aglomerativa**, com um
representante real por cluster.


In [ ]:
# ============================================================
# 5) MÉTODOS DE REDUÇÃO DE CARDINALIDADE
# ============================================================
def farthest_point_sampling(F, n_pontos, seed=42):
    """Seleção gulosa max-min: maximiza a distância mínima entre selecionados."""
    rng = np.random.default_rng(seed)
    n = len(F)
    idx = [rng.integers(n)]
    dist_min = np.linalg.norm(F - F[idx[0]], axis=1)
    for _ in range(n_pontos - 1):
        prox = np.argmax(dist_min)
        idx.append(prox)
        novas = np.linalg.norm(F - F[prox], axis=1)
        dist_min = np.minimum(dist_min, novas)
    return np.sort(np.array(idx))


def k_medoids(F, n_pontos, seed=42, algo_cls=KMeans, **kw):
    """K-Means (ou variante) + ponto real mais próximo do centróide."""
    km = algo_cls(n_clusters=n_pontos, random_state=seed, **kw).fit(F)
    idx = []
    for k in range(n_pontos):
        membros = np.where(km.labels_ == k)[0]
        if len(membros) == 0:
            continue
        centro = km.cluster_centers_[k]
        dists = np.linalg.norm(F[membros] - centro, axis=1)
        idx.append(membros[np.argmin(dists)])
    return np.sort(np.unique(np.array(idx)))


def selecionar_clusterizacao_hierarquica(F, n_pontos, linkage="ward"):
    """Clusterização aglomerativa + medóide aproximado de cada cluster.

    A ligação de Ward opera no espaço normalizado euclidiano. O representante
    é sempre um ponto real da fronteira: o membro mais próximo do centróide do
    respectivo cluster.
    """
    F = np.asarray(F, dtype=float)
    if n_pontos >= len(F):
        return np.arange(len(F), dtype=int)

    modelo = AgglomerativeClustering(
        n_clusters=n_pontos,
        linkage=linkage,
    )
    labels = modelo.fit_predict(F)

    idx = []
    for k in range(n_pontos):
        membros = np.where(labels == k)[0]
        centro = F[membros].mean(axis=0)
        dists = np.linalg.norm(F[membros] - centro, axis=1)
        idx.append(membros[np.argmin(dists)])

    idx = np.sort(np.asarray(idx, dtype=int))
    if len(idx) != n_pontos:
        raise RuntimeError(
            f"Clusterização hierárquica retornou {len(idx)} representantes; "
            f"eram esperados {n_pontos}."
        )
    return idx


def gmm_medoids(F, n_pontos, seed=42, covariance_type="diag"):
    """Mistura Gaussiana + ponto real mais próximo da média."""
    gmm = GaussianMixture(
        n_components=n_pontos,
        covariance_type=covariance_type,
        random_state=seed,
        n_init=3,
        reg_covar=1e-6,
    ).fit(F)
    labels = gmm.predict(F)
    idx = []
    for k in range(n_pontos):
        membros = np.where(labels == k)[0]
        if len(membros) == 0:
            continue
        centro = gmm.means_[k]
        dists = np.linalg.norm(F[membros] - centro, axis=1)
        idx.append(membros[np.argmin(dists)])
    return np.sort(np.unique(np.array(idx)))


def gerar_ref_dirs_uniformes(n_dirs, n_obj, seed=42):
    """Amostra uniforme no simplex M-dimensional (Dirichlet)."""
    rng = np.random.default_rng(seed)
    return rng.dirichlet(np.ones(n_obj), size=n_dirs)


def selecionar_por_ref_dirs(F, n_pontos, seed=42):
    """Associa cada direção de referência ao ponto real mais próximo."""
    n_obj = F.shape[1]
    W = gerar_ref_dirs_uniformes(n_pontos, n_obj, seed=seed)
    W_norm = W / np.linalg.norm(W, axis=1, keepdims=True)
    idx_usados = set()
    idx_selecionados = []
    for w in W_norm:
        proj = F @ w
        perp = F - np.outer(proj, w)
        d_perp = np.linalg.norm(perp, axis=1)
        ordem = np.argsort(d_perp)
        for cand in ordem:
            if cand not in idx_usados:
                idx_usados.add(cand)
                idx_selecionados.append(cand)
                break
    return np.sort(np.array(idx_selecionados))


N_ALVO = len(Fvrf_n)

metodos_reducao = {
    "Farthest-Point Sampling": farthest_point_sampling(Fcnbi_n, N_ALVO),
    "K-Medoids": k_medoids(
        Fcnbi_n, N_ALVO, algo_cls=KMeans, n_init=10
    ),
    "MiniBatchKMeans": k_medoids(
        Fcnbi_n,
        N_ALVO,
        algo_cls=MiniBatchKMeans,
        batch_size=256,
        n_init=10,
    ),
    "GMM": gmm_medoids(Fcnbi_n, N_ALVO),
    "Direções de Referência": selecionar_por_ref_dirs(Fcnbi_n, N_ALVO),
    "Clusterização Hierárquica": selecionar_clusterizacao_hierarquica(
        Fcnbi_n, N_ALVO
    ),
}

print(f"Cardinalidade alvo (VRF-NBI): {N_ALVO}")
for nome, idx in metodos_reducao.items():
    print(f"  {nome}: {len(idx)} pontos selecionados")


In [ ]:
# ============================================================
# TABELA — REDUÇÃO DO CNBI vs. VRF-NBI (mesma cardinalidade)
# ============================================================
linhas_reducao = [
    {
        "metodo": "VRF-NBI (nativo)",
        "n": N_ALVO,
        **metricas_frente(Fvrf_n),
    },
]
indices_reducao = {}

for nome, idx in metodos_reducao.items():
    Fr = Fcnbi_n[idx]
    indices_reducao[nome] = idx
    linhas_reducao.append({
        "metodo": f"CNBI via {nome}",
        "n": len(Fr),
        **metricas_frente(Fr),
    })

df_resultado_reducao = pd.DataFrame(linhas_reducao)
display(df_resultado_reducao)
df_resultado_reducao.to_csv(
    OUT_DIR / "comparacao_reducao_66_metricas.csv", index=False
)


In [ ]:
# ============================================================
# 5b) ROBUSTEZ DA REDUÇÃO — mediana (IQR) sobre múltiplas seeds
# A clusterização hierárquica é determinística e é reportada uma única vez.
# ============================================================
SEEDS_RED = [1, 2, 3, 4, 5]

metodos_estocasticos = {
    "Farthest-Point Sampling": lambda F, n, s: farthest_point_sampling(
        F, n, seed=s
    ),
    "K-Medoids": lambda F, n, s: k_medoids(
        F, n, seed=s, algo_cls=KMeans, n_init=10
    ),
    "MiniBatchKMeans": lambda F, n, s: k_medoids(
        F,
        n,
        seed=s,
        algo_cls=MiniBatchKMeans,
        batch_size=256,
        n_init=10,
    ),
    "GMM": lambda F, n, s: gmm_medoids(F, n, seed=s),
    "Direções de Referência": lambda F, n, s: selecionar_por_ref_dirs(
        F, n, seed=s
    ),
}

linhas_rob = []

for nome, fn in metodos_estocasticos.items():
    vals = []
    for s in SEEDS_RED:
        idx = fn(Fcnbi_n, N_ALVO, s)
        Fr = Fcnbi_n[idx]
        met = metricas_frente(Fr)
        vals.append([
            met["GD"],
            met["IGD"],
            met["HV"],
            met["Spacing"],
            met["Sparsity"],
            len(idx),
        ])

    arr = np.asarray(vals, dtype=float)
    linha = {
        "metodo": nome,
        "n_seeds": len(SEEDS_RED),
        "n_pontos_mediano": float(np.median(arr[:, 5])),
    }
    for j, metrica in enumerate(
        ["GD", "IGD", "HV", "Spacing", "Sparsity"]
    ):
        linha[f"{metrica}_mediana"] = float(np.median(arr[:, j]))
        linha[f"{metrica}_IQR"] = float(
            np.percentile(arr[:, j], 75) - np.percentile(arr[:, j], 25)
        )
    linhas_rob.append(linha)

# Método determinístico: uma execução, IQR igual a zero.
idx_h = selecionar_clusterizacao_hierarquica(Fcnbi_n, N_ALVO)
met_h = metricas_frente(Fcnbi_n[idx_h])
linha_h = {
    "metodo": "Clusterização Hierárquica",
    "n_seeds": 1,
    "n_pontos_mediano": float(len(idx_h)),
}
for metrica in ["GD", "IGD", "HV", "Spacing", "Sparsity"]:
    linha_h[f"{metrica}_mediana"] = float(met_h[metrica])
    linha_h[f"{metrica}_IQR"] = 0.0
linhas_rob.append(linha_h)

df_robustez_reducao = pd.DataFrame(linhas_rob)
display(df_robustez_reducao)
df_robustez_reducao.to_csv(
    OUT_DIR / "robustez_reducao_seeds.csv", index=False
)

print(
    "Compare as medianas e os IQRs dos métodos estocásticos; "
    "a clusterização hierárquica é determinística."
)


## 6) Figuras — redução de cardinalidade

In [ ]:
# ============================================================
# FIGURA 4 — métricas por método de redução, vs. VRF-NBI
# ============================================================
metricas_fig4 = [
    ("GD", f"GD vs P* — mesma cardinalidade ({N_ALVO} pts)"),
    ("IGD", f"IGD vs P* — mesma cardinalidade ({N_ALVO} pts)"),
    ("Spacing", f"Spacing — mesma cardinalidade ({N_ALVO} pts)"),
    ("Sparsity", f"Sparsity — mesma cardinalidade ({N_ALVO} pts)"),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
cores_r = [
    "#4C72B0" if "VRF" in m else "#DD8452"
    for m in df_resultado_reducao["metodo"]
]

for ax, (metrica, titulo) in zip(axes.ravel(), metricas_fig4):
    ax.barh(
        df_resultado_reducao["metodo"],
        df_resultado_reducao[metrica],
        color=cores_r,
    )
    ax.set_title(titulo)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUT_DIR / "fig4_reducao_66_metricas.png", dpi=150)
plt.show()


In [ ]:
# ============================================================
# FIGURA 5 — cobertura: VRF-NBI vs. melhor método de redução
#             segundo o menor IGD
# ============================================================

melhor_metodo_idx = df_resultado_reducao.loc[
    df_resultado_reducao["metodo"] != "VRF-NBI (nativo)",
    "IGD",
].idxmin()

nome_melhor = (
    df_resultado_reducao.loc[melhor_metodo_idx, "metodo"]
    .replace("CNBI via ", "")
)

idx_melhor = indices_reducao[nome_melhor]

df_cnbi_reduzido = pd.DataFrame(
    Fcnbi[idx_melhor],
    columns=RESP,
)

print(f"Melhor método de redução (menor IGD): {nome_melhor}")

# ------------------------------------------------------------
# Organização dos gráficos
# ------------------------------------------------------------
import math

ncols = 2
nrows = math.ceil(len(pares) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(12, 5 * nrows),
)

axes = np.asarray(axes).ravel()

# ------------------------------------------------------------
# Projeções bidimensionais
# ------------------------------------------------------------
for ax, (ox, oy) in zip(axes, pares):

    ax.scatter(
        df_pstar_full[ox],
        df_pstar_full[oy],
        s=6,
        color="lightgray",
        alpha=0.50,
        label="P*",
        zorder=1,
    )

    ax.scatter(
        df_cnbi_reduzido[ox],
        df_cnbi_reduzido[oy],
        s=30,
        color="#DD8452",
        alpha=0.85,
        label=f"CNBI  ({nome_melhor})",
        zorder=2,
    )

    ax.scatter(
        df_vrf_full[ox],
        df_vrf_full[oy],
        s=30,
        color="#4C72B0",
        alpha=0.85,
        label=f"VRF-NBI ({len(df_vrf_full)} pts)",
        zorder=3,
    )

    ax.set_xlabel(ox)
    ax.set_ylabel(oy)
    ax.set_title(f"{ox} vs {oy} ")
    ax.grid(alpha=0.20)
    ax.legend(fontsize=8)

# Remove eixos que sobrarem quando o número de pares não completar a grade
for ax in axes[len(pares):]:
    ax.remove()

plt.tight_layout()

plt.savefig(
    OUT_DIR / "fig5_cobertura_reduzida_igd.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

## 7) Exportação consolidada

In [ ]:
# ============================================================
# 7) EXPORTAÇÃO FINAL — todas as tabelas em um único Excel
# ============================================================
with pd.ExcelWriter(OUT_DIR / "comparacao_fronteiras_completa.xlsx", engine="openpyxl") as writer:
    df_resultado_principal.to_excel(writer, sheet_name="principal_vs_Pstar", index=False)
    if not df_ea_seeds_long.empty:
        df_ea_seeds_long.to_excel(writer, sheet_name="EAs_por_seed", index=False)
    df_resultado_reducao.to_excel(writer, sheet_name="reducao_66_vs_VRF", index=False)
    if "df_robustez_reducao" in globals():
        df_robustez_reducao.to_excel(writer, sheet_name="robustez_reducao_seeds", index=False)

print("Exportação concluída em:", (OUT_DIR / "comparacao_fronteiras_completa.xlsx").resolve())
print("Figuras salvas em:", OUT_DIR.resolve())
